# Text to SQL via prompt engineering

Using Claude (Anthropic) to write SQL queries for the **Life Insurance Death Claims** dataset!

<br><br>
____
<br>
Setting up the database from the li_death_claims CSV:

In [1]:
import sqlite3
import pandas as pd
import os

In [2]:
# SQL schema for the Life Insurance Death Claims table
li_death_claims_schema = """
CREATE TABLE IF NOT EXISTS LIDeathClaims (
  life_insurer TEXT,
  year TEXT,
  claims_pending_start_no INTEGER,
  claims_pending_start_amt REAL,
  claims_intimated_no INTEGER,
  claims_intimated_amt REAL,
  total_claims_no INTEGER,
  total_claims_amt REAL,
  claims_paid_no INTEGER,
  claims_paid_amt REAL,
  claims_repudiated_no INTEGER,
  claims_repudiated_amt REAL,
  claims_rejected_no INTEGER,
  claims_rejected_amt REAL,
  claims_unclaimed_no INTEGER,
  claims_unclaimed_amt REAL,
  claims_pending_end_no INTEGER,
  claims_pending_end_amt REAL,
  claims_paid_ratio_no REAL,
  claims_paid_ratio_amt REAL,
  claims_repudiated_rejected_ratio_no REAL,
  claims_repudiated_rejected_ratio_amt REAL,
  claims_pending_ratio_no REAL,
  claims_pending_ratio_amt REAL,
  category TEXT
);
"""

Putting all data from the CSV into a SQLite database:

In [5]:
# Connect to SQLite database
conn = sqlite3.connect('li_death_claims.db')
cursor = conn.cursor()

# Create the table if it doesn't exist
cursor.execute(li_death_claims_schema)

# Load data from CSV into pandas DataFrame
# Make sure li_death_claims.csv is in the same directory as this notebook
claims_df = pd.read_csv('/media/escanor/extra/programing/gfg/21Days/Submissions/14/li_death_claims.csv')

# Write the data from the DataFrame to the SQLite table
claims_df.to_sql('LIDeathClaims', conn, if_exists='replace', index=False)

# Verify data insertion
cursor.execute("SELECT COUNT(*) FROM LIDeathClaims;")
count = cursor.fetchone()[0]
print(f"Number of rows in LIDeathClaims table: {count}")

# Close the connection
conn.close()

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x94 in position 44299: invalid start byte

Install the Anthropic SDK and set up your API key.

Get your API key from: https://console.anthropic.com/

Store it as an environment variable `ANTHROPIC_API_KEY` or paste it directly below.

In [4]:
!pip install anthropic

  Using cached anyio-4.12.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
Using cached anyio-4.12.1-py3-none-any.whl (113 kB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [anthropic]/9 [anthropic]


Import required modules and set up the Anthropic client:

In [ ]:
import anthropic
import os

In [ ]:
# Set your Anthropic API key
# Option 1: use environment variable (recommended)
# export ANTHROPIC_API_KEY="your-key-here"
#
# Option 2: paste directly (not recommended for shared notebooks)
# os.environ["ANTHROPIC_API_KEY"] = "your-key-here"

anthropic_client = anthropic.Anthropic(
    api_key=os.environ.get("ANTHROPIC_API_KEY")
)

Define the NL-to-SQL system prompt for the Life Insurance Death Claims database:

In [ ]:
prompt = """
### ROLE

You are an expert-level SQLite Database Engineer specializing in Natural Language to SQL (NL2SQL) translation. Your sole function is to convert user questions written in plain English into accurate, efficient, and syntactically correct SQLite queries based on a fixed database schema.

-----

### CONTEXT

You are the core translation engine for an insurance analytics dashboard. This tool allows analysts and non-technical users to query the life insurance death claims database using natural language. The database dialect is always SQLite. Your responses will be executed directly on the database.

The database consists of a single table:

LIDeathClaims table

CREATE TABLE IF NOT EXISTS LIDeathClaims (
  life_insurer TEXT,                          -- Name/abbreviation of the life insurance company
  year TEXT,                                  -- Financial year (e.g. '2021-22')
  claims_pending_start_no INTEGER,            -- Number of claims pending at start of year
  claims_pending_start_amt REAL,              -- Amount (in crores) of claims pending at start of year
  claims_intimated_no INTEGER,               -- Number of new claims intimated during the year
  claims_intimated_amt REAL,                 -- Amount of new claims intimated
  total_claims_no INTEGER,                   -- Total claims (pending start + intimated)
  total_claims_amt REAL,                     -- Total claims amount
  claims_paid_no INTEGER,                    -- Number of claims paid
  claims_paid_amt REAL,                      -- Amount of claims paid
  claims_repudiated_no INTEGER,              -- Number of claims repudiated (rejected after investigation)
  claims_repudiated_amt REAL,                -- Amount of repudiated claims
  claims_rejected_no INTEGER,                -- Number of claims rejected
  claims_rejected_amt REAL,                  -- Amount of rejected claims
  claims_unclaimed_no INTEGER,               -- Number of unclaimed/lapsed claims
  claims_unclaimed_amt REAL,                 -- Amount of unclaimed claims
  claims_pending_end_no INTEGER,             -- Number of claims pending at end of year
  claims_pending_end_amt REAL,               -- Amount of claims pending at end of year
  claims_paid_ratio_no REAL,                 -- Claim settlement ratio by number (0 to 1)
  claims_paid_ratio_amt REAL,                -- Claim settlement ratio by amount (0 to 1)
  claims_repudiated_rejected_ratio_no REAL,  -- Repudiation+rejection ratio by number
  claims_repudiated_rejected_ratio_amt REAL, -- Repudiation+rejection ratio by amount
  claims_pending_ratio_no REAL,              -- Pending ratio by number
  claims_pending_ratio_amt REAL,             -- Pending ratio by amount
  category TEXT                              -- Type of claim (e.g. 'Individual Death Claims')
);

-----

### TASK

Your task is to receive a user's question in natural language and convert it into a single, executable SQLite query. Follow these steps meticulously:

1. Analyze the User's Query: Deconstruct the user's question to understand their core intent. Identify the specific data, conditions, aggregations (like SUM, COUNT, AVG, MAX, MIN), and ordering they are asking for.
2. Map to the Schema: Map the entities from the user's query to the appropriate columns in the LIDeathClaims table. Use only the columns provided in the schema.
3. Construct the SQLite Query: Write a clean and efficient SELECT statement that is syntactically correct for SQLite. Ensure all column names are accurate.
   - **Important:** If combining multiple SELECT statements (e.g. highest and lowest), either use MAX()/MIN() or wrap each SELECT in parentheses before applying UNION or UNION ALL. Avoid using ORDER BY before UNION.
   - Ratios are stored as decimals (0 to 1). To express as a percentage, multiply by 100.
4. Handle Ambiguity: If the user's query is vague, ambiguous, or lacks the necessary information to create a precise query, do not guess. Instead, formulate a specific, targeted question to ask the user for the missing information.

-----

### CONSTRAINTS

- Read-Only Operations: You must only generate SELECT queries. Never generate INSERT, UPDATE, DELETE, DROP, or any other data-modifying statements.
- Adhere Strictly to Schema: Only use the LIDeathClaims table and its defined columns. Do not invent or assume the existence of any other tables or columns.
- No Explanations: Do not add any conversational text or explanations about the query you generate. Your output must strictly follow the specified format.
- Single Query Only: The final output must be a single, complete, and executable SQL query.
- Handle Impossibility: If a request is impossible to fulfill with the given schema, state clearly that the request cannot be completed and briefly explain why.

-----

### EXAMPLES

Example 1: Simple Lookup
User Query: "Show me all records for HDFC Life"
Expected Output:
{
  "status": "success",
  "response": "SELECT * FROM LIDeathClaims WHERE life_insurer = 'HDFC Life';"
}

Example 2: Ranking by settlement ratio
User Query: "Which insurer had the highest claim settlement ratio by number in 2021-22?"
Expected Output:
{
  "status": "success",
  "response": "SELECT life_insurer, claims_paid_ratio_no FROM LIDeathClaims WHERE year = '2021-22' ORDER BY claims_paid_ratio_no DESC LIMIT 1;"
}

Example 3: Multi-extreme query (highest and lowest)
User Query: "Which insurer had the highest and lowest claim settlement ratio by number?"
Expected Output:
{
  "status": "success",
  "response": "SELECT life_insurer, claims_paid_ratio_no FROM LIDeathClaims WHERE claims_paid_ratio_no = (SELECT MAX(claims_paid_ratio_no) FROM LIDeathClaims) UNION ALL SELECT life_insurer, claims_paid_ratio_no FROM LIDeathClaims WHERE claims_paid_ratio_no = (SELECT MIN(claims_paid_ratio_no) FROM LIDeathClaims);"
}

Example 4: Ambiguous Query
User Query: "Show me recent claims data"
Expected Output:
{
  "status": "clarification_needed",
  "response": "Could you please specify which year you are interested in? For example, '2021-22' or '2022-23'."
}

Example 5: Impossible Query
User Query: "Which insurer had the most complaints?"
Expected Output:
{
  "status": "error",
  "response": "I cannot answer this question as the database does not contain information about customer complaints."
}

-----

### OUTPUT FORMAT

Your final response must be a single JSON object with two keys:

1. "status": A string with one of three possible values: "success", "clarification_needed", or "error".
2. "response":
   - If status is "success", this will be a string containing the complete SQLite query.
   - If status is "clarification_needed", this will be a string containing the clarifying question for the user.
   - If status is "error", this will be a string explaining why the query could not be generated.
"""

In [ ]:
import json

def get_sql_query(client, prompt, user_query):

    contents = f"""
    {prompt}

    Here's the user query in english you need to work on:
    {user_query}
    """

    message = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        messages=[
            {"role": "user", "content": contents}
        ]
    )

    # Print token usage
    print(f"Input Token Count: {message.usage.input_tokens}")
    print(f"Output Token Count: {message.usage.output_tokens}")

    response_text = message.content[0].text
    output = json.loads(response_text.replace('```json', '').replace('```', '').strip())

    return output


In [ ]:
import sqlite3
import pandas as pd

def execute_query(query, db_name='li_death_claims.db'):

    conn = None
    try:
        # Connect to the database
        conn = sqlite3.connect(db_name)
        cursor = conn.cursor()

        # Execute the query
        print(f"\nExecuting query on '{db_name}':\n{query}")
        cursor.execute(query)

        # Fetch all results
        results = cursor.fetchall()

        # Get column names from the cursor description
        columns = [description[0] for description in cursor.description]

        # Format results as a dataframe for easier use
        results_as_dict = [dict(zip(columns, row)) for row in results]
        results_df = pd.DataFrame(results_as_dict)

        print("Query executed successfully.")
        return results_df

    except sqlite3.Error as e:
        print(f"Database error executing query: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None
    finally:
        if conn:
            conn.close()

In [ ]:
def text2sql(client, prompt, user_query):
    output = get_sql_query(client, prompt, user_query)
    if output['status'] == 'success':
        results = execute_query(output['response'])
        return results
    return output

---
## Example Queries

Let's try some natural language questions against the Life Insurance Death Claims data!

In [ ]:
# Example 1: Count of records per insurer
text2sql(anthropic_client, prompt, "Show me the total number of records for each life insurer")

In [ ]:
# Example 2: Best settlement ratio
text2sql(anthropic_client, prompt, "Which insurer had the highest claim settlement ratio by number of claims in 2021-22?")

In [ ]:
# Example 3: Total claims paid
text2sql(anthropic_client, prompt, "What is the total amount paid out in claims across all insurers?")

In [ ]:
# Example 4: Average settlement ratio by insurer
text2sql(anthropic_client, prompt, "What is the average claim settlement ratio by number for each insurer?")

In [ ]:
# Example 5: Insurers with most pending claims at year end
text2sql(anthropic_client, prompt, "Which insurer had the highest and lowest claim settlement ratio by number?")

In [ ]:
# Example 6: Total claims intimated
text2sql(anthropic_client, prompt, "How many total claims were intimated across all insurers in 2021-22?")

In [ ]:
# Example 7: Repudiation comparison
text2sql(anthropic_client, prompt, "Show me the insurer with the highest repudiation and rejection ratio by number of claims")

In [ ]:
# Example 8: Error handling - impossible query
text2sql(anthropic_client, prompt, "Which insurer had the highest customer satisfaction score?")

In [ ]:
# Example 9: Clarification needed
text2sql(anthropic_client, prompt, "Show me recent data")